# Preliminary experiments
See tex/final/tesi.pdf, Appendix B. Every figure in the appendix is produced here, in the order it appears in the text, and saved to `out_dir`.

Code collected from `preliminary/noise_structure.ipynb` and `preliminary/more_complex_fun.ipynb`.

In [ ]:
%run base.ipynb

In [ ]:
# MODEL
mymodel = DecisionTreeClassifier

# PARAMETERS
supp_size = 10_000
data_size_test = 200_000
data_size_tr = 50
num_iter = 100
# bernoulli noise
prob = 0.8
# number of training sets used to calibrate the gaussian noise
num_samples = 100

from functools import partial
from contextlib import contextmanager

# figures used in the thesis
out_dir = '../tex/final/outputs/'

@contextmanager
def save_shown_figure(path):
    # explore_data() and find_sigma_star() call plt.show() themselves:
    # save the current figure right before it is shown
    show = plt.show
    def show_and_save(*args, **kwargs):
        plt.gcf().savefig(path, bbox_inches='tight')
        show(*args, **kwargs)
    plt.show = show_and_save
    try:
        yield
    finally:
        plt.show = show

def train_test_plot(f, sigma, path, noise_type='gaussian', p=prob):
    # train vs test error only (first plot of plots()), averaged over num_iter training sets
    x_test = sample_x(data_size_test, supp_size)
    if noise_type == 'gaussian':
        y_test = sample_y(x=x_test, f=f, sigma=sigma)
    elif noise_type == 'bernoulli':
        y_test = sample_y_bernoulli(x=x_test, f=f, p=p)

    hat_Rs, Rs, gaps, std_hat_Rs, std_Rs, std_gaps, *_ = main_loop(
        x_test, y_test, n=data_size_tr, sigma=sigma, T=num_iter, model=mymodel,
        f=f, noise_type=noise_type, p=p, N=supp_size)

    fig, ax = plt.subplots(figsize=(10, 5))
    plots(hat_Rs, Rs, gaps, std_hat_Rs=std_hat_Rs, std_Rs=std_Rs, std_gaps=std_gaps,
          n=data_size_tr, sigma=sigma, T=num_iter, model=mymodel,
          bound_name="unionbound", bound_params={},
          noise_type=noise_type, p=p, two=False, three=False, axes=[ax])
    ax.set_title('train vs test error')
    plt.tight_layout()
    fig.savefig(out_dir + path, bbox_inches='tight')
    plt.show()

## B.1 Noise structure
Labels generated with the centered parabola. We look for the $\sigma$ such that gaussian noise flips on average the same fraction of labels as bernoulli noise with $p = 0.8$ (20%), then compare train and test errors under the two noise structures.

In [ ]:
with save_shown_figure(out_dir + 'app_sigmastar_parabola.png'):
    sigma_star_parabola = find_sigma_star(f=centered_parabola, p=prob, n=data_size_tr, N=supp_size, num_samples=num_samples)
print(f'sigma* (parabola) = {sigma_star_parabola}')

In [ ]:
# bernoulli noise, p = 0.8
train_test_plot(centered_parabola, sigma=None, path='app_parabola_bernoulli.png', noise_type='bernoulli')

In [ ]:
# gaussian noise, sigma = sigma*
train_test_plot(centered_parabola, sigma=sigma_star_parabola, path='app_parabola_gauss.png')

## B.2 Data-generating function
### Parabola

In [ ]:
with save_shown_figure(out_dir + 'app_explore_parabola.png'):
    explore_data(f=centered_parabola, n=data_size_tr, N=supp_size, sigmas=(10**(-4), 1, sigma_star_parabola, 10))

### Sine
$m$ is the number of crossings of the $x$ axis in $(0, 10)$. The noise is a bit lower than for the parabola, $\sigma = 3$: the sine has amplitude 5, so it spends more of the domain close to $f = 0$ and the same $\sigma$ flips more labels.

In [ ]:
# noise used for the sine until it is calibrated below
noise_std = 3

with save_shown_figure(out_dir + 'app_explore_sine.png'):
    explore_data(f=partial(sine, m=10), n=data_size_tr, N=supp_size, sigmas=(10**(-4), 1, noise_std, 5))

In [ ]:
for num_crossings in [10, 100, 1_000]:
    train_test_plot(partial(sine, m=num_crossings), sigma=noise_std, path=f'app_sine_m{num_crossings}.png')

### Learnability threshold
Fix the ratio $r = m/n$ and look for the threshold above which the test error saturates at 0.5 (as good as guessing).

In [ ]:
ratios = [20, 15, 10, 5, 2, 1]

fig, axs = plt.subplots(3, 2, figsize=(12, 14))
fig.suptitle(r'train vs test error for different $r = m/n$ ratios', fontsize=13)

for ax, r in zip(axs.flat, ratios):
    num_crossings = round(r * data_size_tr)
    myfun = partial(sine, m=num_crossings)

    # build test set
    x_test = sample_x(data_size_test, supp_size)
    y_test = sample_y(x=x_test, f=myfun, sigma=noise_std)

    # build models and do plots (only the train vs test error plot)
    hat_Rs, Rs, gaps, std_hat_Rs, std_Rs, std_gaps, *_ = main_loop(
        x_test, y_test, n=data_size_tr, sigma=noise_std, T=num_iter, model=mymodel, f=myfun, N=supp_size)
    plots(hat_Rs, Rs, gaps, std_hat_Rs=std_hat_Rs, std_Rs=std_Rs, std_gaps=std_gaps,
          n=data_size_tr, sigma=noise_std, T=num_iter, model=mymodel,
          bound_name="unionbound", bound_params={}, two=False, three=False, axes=[ax])
    ax.set_title(f'r = {r}  (m = {num_crossings})\ntrain vs test error')

plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(out_dir + 'app_ratio_sweep.png', bbox_inches='tight')
plt.show()

### Noise calibration on the sine
From now on $m = 10$. The $\sigma^*$ calibrated on the parabola does not flip 20% of the labels on the sine, so we recalibrate it.

In [ ]:
num_crossings = 10
myfun = partial(sine, m=num_crossings)

# fraction of labels that disagree with the noiseless sign(f(x)), for each sample
frac_gauss = []; frac_bern = []
for _ in range(num_samples):
    x = sample_x(data_size_tr, supp_size)
    true_y = np.where(myfun(x) >= 0, 1, -1)
    frac_gauss.append(np.mean(sample_y(x, myfun, sigma=sigma_star_parabola) != true_y))
    frac_bern.append(np.mean(sample_y_bernoulli(x, myfun, p=prob) != true_y))

print(f'gaussian,  sigma* (parabola) = {sigma_star_parabola}: {np.mean(frac_gauss):.3f} ± {np.std(frac_gauss):.3f} misclassified')
print(f'bernoulli, p = {prob}:{" " * 17}{np.mean(frac_bern):.3f} ± {np.std(frac_bern):.3f} misclassified')

# distribution over the samples of the fraction misclassified, for the two noise structures
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(0, 0.5, 26)
ax.hist(frac_gauss, bins=bins, color='#00CED1', alpha=0.6, label=rf'gaussian, $\sigma$ = {sigma_star_parabola}')
ax.hist(frac_bern,  bins=bins, color='#FF007F', alpha=0.6, label=rf'bernoulli, $p$ = {prob}')
ax.axvline(np.mean(frac_gauss), color='#00CED1', linestyle='--', linewidth=1.5)
ax.axvline(np.mean(frac_bern),  color='#FF007F', linestyle='--', linewidth=1.5)
ax.set_title(f'fraction of misclassified inputs over {num_samples} samples of size {data_size_tr} ($m$ = {num_crossings})')
ax.set_xlabel('fraction misclassified'); ax.set_ylabel('count')
ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig(out_dir + 'app_misclassified_sine.png', bbox_inches='tight')
plt.show()

In [ ]:
# recalibrate sigma* on the sine
with save_shown_figure(out_dir + 'app_sigmastar_sine.png'):
    sigma_star = find_sigma_star(f=myfun, p=prob, n=data_size_tr, N=supp_size, num_samples=num_samples)
print(f'sigma* (sine) = {sigma_star}')

### Sample size needed for a simple function
Sine with $m = 2$ (same classification pattern as the parabola). Smallest $n$ for which the test error gets within `tol` of the Bayes risk on average, at depth 2 and 3.

In [ ]:
# PARAMETERS (on top of the defaults)
depths = [2, 3]
data_sizes = [10, 20, 50, 100, 200, 500, 1_000, 2_000, 5_000]
tol = 0.01

def simple_function(f, sigma, num_crossings, path):
    # build test set, and its Bayes risk (labels flipped by the noise w.r.t. sign(f(x)))
    x_test = sample_x(data_size_test, supp_size)
    y_test = sample_y(x=x_test, f=f, sigma=sigma)
    bayes_risk = np.mean(y_test != np.where(f(x_test) >= 0, 1, -1))

    # E[R] for each depth and training set size, averaged over num_iter training sets
    E_Rs = {d: [] for d in depths}
    for n in data_sizes:
        Rs = {d: [] for d in depths}
        for _ in range(num_iter):
            x = sample_x(n, supp_size)
            y = sample_y(x, f, sigma=sigma)
            for d in depths:
                clf = mymodel(max_depth=d, random_state=42).fit(x.reshape(-1, 1), y)
                Rs[d].append(1 - clf.score(x_test.reshape(-1, 1), y_test))
        for d in depths:
            E_Rs[d].append(np.mean(Rs[d]))

    # smallest n for which E[R] is within tol of the Bayes risk
    n_sufficient = {d: next((n for n, e in zip(data_sizes, E_Rs[d]) if e - bayes_risk <= tol), None) for d in depths}
    print(f'Bayes risk: {bayes_risk:.3f}')
    for d in depths:
        print(f'depth {d}: n = {n_sufficient[d]} is sufficient')

    fig, ax = plt.subplots(figsize=(10, 5))
    for d, color in zip(depths, ['#FF007F', '#00CED1']):
        ax.plot(data_sizes, E_Rs[d], color=color, marker='o', label=rf'$E[R]$, depth {d}')
    ax.axhline(bayes_risk, color='gray', linestyle='--', linewidth=1.2, label='Bayes risk')
    ax.axhline(bayes_risk + tol, color='gray', linestyle=':', linewidth=1.2, label=f'Bayes risk + {tol}')
    ax.set_xscale('log')
    ax.set_title(rf'test error against training set size ($m$ = {num_crossings}, $\sigma$ = {sigma})')
    ax.set_xlabel('n'); ax.set_ylabel('error')
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.savefig(out_dir + path, bbox_inches='tight')
    plt.show()

num_crossings = 2
myfun = partial(sine, m=num_crossings)

# little noise
simple_function(myfun, sigma=1, num_crossings=num_crossings, path='app_simple_function_sigma1.png')

In [ ]:
# noise calibrated on the sine above
simple_function(myfun, sigma=sigma_star, num_crossings=num_crossings, path='app_simple_function.png')